# Causal Verification

**Survey Reference:** Section 8.3 - Causal Verification and Editing

## Overview

Discovering a circuit is not enough—we need to **verify** that it actually implements the hypothesized computation. Causal verification uses interventions to test circuit hypotheses.

### Key Questions

1. Is the circuit **sufficient?** (Does it reproduce the behavior?)
2. Is the circuit **necessary?** (Does ablating it break the behavior?)
3. Is the circuit **faithful?** (Does our description match its computation?)

## Learning Objectives

1. Understand causal scrubbing and resample ablation
2. Implement sufficiency and necessity tests
3. Verify circuit hypotheses with interventions
4. Identify failure modes of circuit claims

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Callable, Optional

device = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Sufficiency Testing

A circuit is **sufficient** if running only those components (with others ablated) reproduces the behavior.

$$\text{Sufficiency} = \frac{\text{Metric}(\text{circuit only})}{\text{Metric}(\text{full model})}$$

In [ ]:
def test_sufficiency(
    model,
    inputs,
    metric_fn: Callable,
    circuit_components: list[str],
    ablation_value: str = "zero",  # "zero", "mean", "resample"
):
    """
    Test if a circuit is sufficient to produce the behavior.
    
    Args:
        model: The model to test
        inputs: Test inputs
        metric_fn: Function measuring the behavior
        circuit_components: Components in the hypothesized circuit
        ablation_value: How to ablate non-circuit components
    """
    # Full model performance
    # full_metric = metric_fn(model(inputs))
    
    # Circuit-only performance (ablate everything else)
    # circuit_metric = run_with_ablations(
    #     model, inputs,
    #     ablate=all_components - circuit_components,
    #     ablation_type=ablation_value
    # )
    
    # sufficiency = circuit_metric / full_metric
    # return sufficiency
    pass

## 2. Necessity Testing

A circuit is **necessary** if ablating it significantly degrades the behavior.

$$\text{Necessity} = 1 - \frac{\text{Metric}(\text{circuit ablated})}{\text{Metric}(\text{full model})}$$

In [ ]:
def test_necessity(
    model,
    inputs,
    metric_fn: Callable,
    circuit_components: list[str],
    ablation_value: str = "zero",
):
    """
    Test if a circuit is necessary for the behavior.
    
    Ablates the circuit and measures performance drop.
    """
    # Full model performance
    # full_metric = metric_fn(model(inputs))
    
    # Performance with circuit ablated
    # ablated_metric = run_with_ablations(
    #     model, inputs,
    #     ablate=circuit_components,
    #     ablation_type=ablation_value
    # )
    
    # necessity = 1 - (ablated_metric / full_metric)
    # return necessity
    pass

## 3. Causal Scrubbing

**Causal scrubbing** is a rigorous verification method that:

1. Defines a **computational graph** hypothesis
2. For each node, checks if activations are used "as hypothesized"
3. Replaces activations with resampled values when hypothesis allows

If the hypothesis is correct, resampling shouldn't hurt performance.

In [ ]:
class CircuitHypothesis:
    """
    A hypothesis about how a circuit computes.
    
    Specifies which components depend on which input features.
    """
    
    def __init__(self):
        # Mapping: component -> list of input features it depends on
        self.dependencies = {}
    
    def add_dependency(self, component: str, input_features: list[str]):
        """Specify that component depends on input_features."""
        self.dependencies[component] = input_features
    
    def can_resample(self, component: str, feature_changed: str) -> bool:
        """
        Check if component can be resampled when feature changes.
        
        If component doesn't depend on feature, resampling is valid.
        """
        if component not in self.dependencies:
            return True
        return feature_changed not in self.dependencies[component]

In [ ]:
def causal_scrubbing_score(
    model,
    inputs,
    metric_fn: Callable,
    hypothesis: CircuitHypothesis,
):
    """
    Compute causal scrubbing score for a circuit hypothesis.
    
    High score = hypothesis matches actual computation.
    """
    # Original performance
    # original_metric = metric_fn(model(inputs))
    
    # For each component not in hypothesis, resample activation
    # For each component in hypothesis, keep original
    # scrubbed_metric = run_with_scrubbing(model, inputs, hypothesis)
    
    # Score = scrubbed / original
    # return scrubbed_metric / original_metric
    pass

## 4. Resample Ablation

Different ablation methods test different hypotheses:

| Method | Description | Tests |
|--------|-------------|-------|
| Zero ablation | Set activations to 0 | If component contributes additively |
| Mean ablation | Set to dataset mean | If component adds input-specific info |
| Resample ablation | Replace with different input's activation | If component uses specific input |

In [ ]:
def resample_ablation(
    activations: torch.Tensor,
    batch_size: int,
):
    """
    Resample activations by shuffling within batch.
    
    Each sample gets a different sample's activation.
    """
    perm = torch.randperm(batch_size)
    # Avoid self-assignment
    while (perm == torch.arange(batch_size)).any():
        perm = torch.randperm(batch_size)
    return activations[perm]

def mean_ablation(activations: torch.Tensor):
    """Replace with mean activation."""
    return activations.mean(dim=0, keepdim=True).expand_as(activations)

def zero_ablation(activations: torch.Tensor):
    """Replace with zeros."""
    return torch.zeros_like(activations)

## 5. Activation Editing

Beyond verification, we can **edit** activations to change behavior:

- **Patch in desired activations** to induce behavior
- **Ablate unwanted components** to remove behavior
- **Add steering vectors** to shift behavior

In [ ]:
def activation_editing(
    model,
    input_ids,
    component: str,
    edit_fn: Callable[[torch.Tensor], torch.Tensor],
):
    """
    Run model with an activation edit at a specific component.
    
    Args:
        edit_fn: Function that takes original activation and returns edited
    """
    # Hook to modify activation
    def edit_hook(module, input, output):
        return edit_fn(output)
    
    # TODO: Register hook at component, run model, remove hook
    pass

## 6. Common Verification Pitfalls

### Pitfall 1: Insufficiency ≠ Wrong Circuit
A circuit might be incomplete rather than wrong.

### Pitfall 2: Ablation Side Effects
Ablating one component can affect others in unexpected ways.

### Pitfall 3: Distribution Shift
Ablated activations may be out-of-distribution for downstream components.

In [ ]:
def check_ablation_distribution(
    original_acts: torch.Tensor,
    ablated_acts: torch.Tensor,
):
    """
    Check if ablated activations are in-distribution.
    """
    # Compare statistics
    orig_mean = original_acts.mean().item()
    orig_std = original_acts.std().item()
    abl_mean = ablated_acts.mean().item()
    abl_std = ablated_acts.std().item()
    
    print(f"Original: mean={orig_mean:.4f}, std={orig_std:.4f}")
    print(f"Ablated:  mean={abl_mean:.4f}, std={abl_std:.4f}")
    
    # Large shifts may indicate OOD issues
    mean_shift = abs(orig_mean - abl_mean) / (orig_std + 1e-6)
    print(f"Mean shift (in stds): {mean_shift:.2f}")

## Key Takeaways

1. **Sufficiency:** Can the circuit alone reproduce the behavior?
2. **Necessity:** Does ablating the circuit break the behavior?
3. **Causal scrubbing** provides rigorous hypothesis testing
4. **Ablation method matters:** Zero, mean, and resample test different things
5. **Watch for pitfalls:** Incompleteness, side effects, distribution shift

## Next Steps

→ **05_intermediate_computation/01_logit_lens.ipynb**: Understand how predictions evolve through layers